# CHAPTER 7 - Data Cleaning and Preparation

During the course of doing data analysis and modeling, a significant amount of time
is spent on data preparation: loading, cleaning, transforming, and rearranging. Such
tasks are often reported to take up 80% or more of an analyst’s time. Sometimes the
way that data is stored in files or databases is not in the right format for a particular
task. Many researchers choose to do ad hoc processing of data from one form to
another using a general-purpose programming language, like Python, Perl, R, or Java,
or Unix text-processing tools like sed or awk. Fortunately, pandas, along with the
built-in Python language features, provides you with a high-level, flexible, and fast set
of tools to enable you to manipulate data into the right form.

If you identify a type of data manipulation that isn’t anywhere in this book or elsewhere
in the pandas library, feel free to share your use case on one of the Python
mailing lists or on the pandas GitHub site. Indeed, much of the design and implementation
of pandas has been driven by the needs of real-world applications.

In this chapter I discuss tools for missing data, duplicate data, string manipulation,
and some other analytical data transformations. In the next chapter, I focus on combining
and rearranging datasets in various ways.

## Basic Imports and Set ups

In [2]:
import pandas as pd
import numpy as np
import os
from faker import Faker

This is a function to render dataframes as tables in the Jupyter Notebook.

In [3]:
import pandas as pd
from IPython.display import display, HTML

def render_book_table(title, columns, rows):
    """
    Render a documentation-style table in Jupyter.

    Parameters
    ----------
    title : str
        Title displayed above the table
    columns : list
        Column names
    rows : list of lists
        Table data rows
    """

    table_data = pd.DataFrame(rows, columns=columns)

    styled_table = (
        table_data.style
        .hide(axis="index")
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#f2f2f2"),
                       ("font-weight", "bold"),
                       ("text-align", "center"),
                       ("border", "1px solid #999"),
                       ("padding", "8px")]},

            {"selector": "td",
             "props": [("border", "1px solid #999"),
                       ("padding", "8px"),
                       ("white-space", "normal"),
                       ("text-align", "left")]},

            {"selector": "table",
             "props": [("border-collapse", "collapse"),
                       ("width", "100%")]}
        ])
    )

    display(HTML(f"<h3>{title}</h3>"))
    display(styled_table)



## 7.1 Handling Missing Data

Missing data occurs commonly in many data analysis applications. One of the goals
of pandas is to make working with missing data as painless as possible. For example,
all of the descriptive statistics on pandas objects exclude missing data by default.

The way that missing data is represented in pandas objects is somewhat imperfect,
but it is functional for a lot of users. For numeric data, pandas uses the floating-point
value `NaN` (Not a Number) to represent missing data. We call this a *sentinel* value that
can be easily detected:

In [7]:
string_data = pd.Series(['aardvark', 'artichoke', np.nan, 'avocado'])

In [8]:
string_data.isnull()

0    False
1    False
2     True
3    False
dtype: bool

In pandas, we’ve adopted a convention used in the R programming language by referring
to missing data as NA, which stands for not available. In statistics applications,
NA data may either be data that does not exist or that exists but was not observed
(through problems with data collection, for example). When cleaning up data for
analysis, it is often important to do analysis on the missing data itself to identify data
collection problems or potential biases in the data caused by missing data.

The built-in Python `None` value is also treated as NA in object arrays:

In [10]:
string_data[0] = None

In [11]:
string_data.isnull()


0     True
1    False
2     True
3    False
dtype: bool

There is work ongoing in the pandas project to improve the internal details of how
missing data is handled, but the user API functions, like `pandas.isnull`, abstract
away many of the annoying details. See Table 7-1 for a list of some functions related
to missing data handling.

In [12]:
columns = ["Argument", "Description"]

rows = [
["dropna","Filter axis labels based on whether values for each label have missing data, with varying thresholds for how much missing data to tolerate"],
["fillna","Fill in missing data with some value or using an interpolation method such as 'ffill' or 'bfill'"],
["isnull","Return boolean values indicating which values are missing/NA"],
["notnull","Negation of isnull"]
]

render_book_table(
    "Table 7-1. NA Handling Methods",
    columns,
    rows
)

Argument,Description
dropna,"Filter axis labels based on whether values for each label have missing data, with varying thresholds for how much missing data to tolerate"
fillna,Fill in missing data with some value or using an interpolation method such as 'ffill' or 'bfill'
isnull,Return boolean values indicating which values are missing/NA
notnull,Negation of isnull


### Filtering Out Missing Data

There are a few ways to filter out missing data. While you always have the option to
do it by hand using `pandas.isnull` and boolean indexing, the `dropna` can be helpful.
On a Series, it returns the Series with only the non-null data and index values:

In [13]:
from numpy import nan as NA

In [14]:
data = pd.Series([1, NA, 3.5, NA, 7])

In [15]:
data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

In [16]:
data.dropna()

0    1.0
2    3.5
4    7.0
dtype: float64

---

With DataFrame objects, things are a bit more complex. You may want to drop rows
or columns that are all NA or only those containing any NAs. `dropna` by default drops
any row containing a missing value:

In [18]:
data = pd.DataFrame([[1., 6.5, 3.], [1., NA, NA],[NA, NA, NA], [NA, 6.5, 3.]])

In [19]:
cleaned = data.dropna()

In [20]:
data

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


In [21]:
cleaned

,0,1,2
0,1.0,6.5,3.0


Passing `how='all'` will only drop rows that are all NA:

In [22]:
data.dropna(how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
3,NaN,6.5,3.0


To drop columns in the same way, pass `axis=1`:

In [23]:
data[4] = NA

In [24]:
data

,0,1,2,4
0,1.0,6.5,3.0,NaN
1,1.0,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,6.5,3.0,NaN


In [25]:
data.dropna(axis=1, how='all')

,0,1,2
0,1.0,6.5,3.0
1,1.0,NaN,NaN
2,NaN,NaN,NaN
3,NaN,6.5,3.0


A related way to filter out DataFrame rows tends to concern time series data. Suppose
you want to keep only rows containing a certain number of observations. You can
indicate this with the `thresh` argument:

In [26]:
df = pd.DataFrame(np.random.randn(7, 3))

In [27]:
df.iloc[:4, 1] = NA

In [28]:
df.iloc[:2, 2] = NA

In [29]:
df

,0,1,2
0,-1.240294,NaN,NaN
1,0.012410,NaN,NaN
2,-1.020198,NaN,-0.719319
3,1.032250,NaN,1.899404
4,0.717973,-1.587971,1.937255
5,-1.392681,0.329188,1.288380
6,-1.029091,-0.013350,-1.874092


In [30]:
df.dropna()

,0,1,2
4,0.717973,-1.587971,1.937255
5,-1.392681,0.329188,1.288380
6,-1.029091,-0.013350,-1.874092


In [31]:
df.dropna(thresh=2)

,0,1,2
2,-1.020198,NaN,-0.719319
3,1.032250,NaN,1.899404
4,0.717973,-1.587971,1.937255
5,-1.392681,0.329188,1.288380
6,-1.029091,-0.013350,-1.874092


### Filling In Missing Data

Rather than filtering out missing data (and potentially discarding other data along with it), you may want to fill in the “holes” in any number of ways. For most purposes,
the `fillna` method is the workhorse function to use. Calling `fillna` with a
constant replaces missing values with that value:

In [33]:
df.fillna(0)

,0,1,2
0,-1.240294,0.000000,0.000000
1,0.012410,0.000000,0.000000
2,-1.020198,0.000000,-0.719319
3,1.032250,0.000000,1.899404
4,0.717973,-1.587971,1.937255
5,-1.392681,0.329188,1.288380
6,-1.029091,-0.013350,-1.874092


Calling `fillna` with a dict, you can use a different fill value for each column:

In [35]:
df.fillna({1: 0.5, 2: 0})

,0,1,2
0,-1.240294,0.500000,0.000000
1,0.012410,0.500000,0.000000
2,-1.020198,0.500000,-0.719319
3,1.032250,0.500000,1.899404
4,0.717973,-1.587971,1.937255
5,-1.392681,0.329188,1.288380
6,-1.029091,-0.013350,-1.874092


fillna returns a new object, but you can modify the existing object in-place:

In [36]:
_ = df.fillna(0, inplace=True)

In [37]:
df

,0,1,2
0,-1.240294,0.000000,0.000000
1,0.012410,0.000000,0.000000
2,-1.020198,0.000000,-0.719319
3,1.032250,0.000000,1.899404
4,0.717973,-1.587971,1.937255
5,-1.392681,0.329188,1.288380
6,-1.029091,-0.013350,-1.874092


---

The same interpolation methods available for reindexing can be used with `fillna`:

In [38]:
df = pd.DataFrame(np.random.randn(6, 3))

In [39]:
df.iloc[2:, 1] = NA

In [40]:
df.iloc[4:, 2] = NA

In [41]:
df

,0,1,2
0,-0.239177,-0.582410,-0.137481
1,-0.173137,-2.772007,0.811962
2,-0.057987,NaN,-0.234244
3,0.647089,NaN,0.599109
4,-0.076055,NaN,NaN
5,-0.882121,NaN,NaN


In [42]:
df.fillna(method='ffill')

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_39416\1193302488.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill')


,0,1,2
0,-0.239177,-0.582410,-0.137481
1,-0.173137,-2.772007,0.811962
2,-0.057987,-2.772007,-0.234244
3,0.647089,-2.772007,0.599109
4,-0.076055,-2.772007,0.599109
5,-0.882121,-2.772007,0.599109


In [43]:
df.ffill()

,0,1,2
0,-0.239177,-0.582410,-0.137481
1,-0.173137,-2.772007,0.811962
2,-0.057987,-2.772007,-0.234244
3,0.647089,-2.772007,0.599109
4,-0.076055,-2.772007,0.599109
5,-0.882121,-2.772007,0.599109


In [46]:
df.fillna(method='ffill', limit=2)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_39416\2719175769.py:1: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', limit=2)


,0,1,2
0,-0.239177,-0.582410,-0.137481
1,-0.173137,-2.772007,0.811962
2,-0.057987,-2.772007,-0.234244
3,0.647089,-2.772007,0.599109
4,-0.076055,NaN,0.599109
5,-0.882121,NaN,0.599109


With `fillna` you can do lots of other things with a little creativity. For example, you
might pass the mean or median value of a Series:

In [44]:
data = pd.Series([1., NA, 3.5, NA, 7])

In [45]:
data.fillna(data.mean())

0    1.000000
1    3.833333
2    3.500000
3    3.833333
4    7.000000
dtype: float64

See Table 7-2 for a reference on `fillna`.



In [47]:
columns = ["Argument", "Description"]

rows = [
["value","Scalar value or dict-like object to use to fill missing values"],
["method","Interpolation method; typically 'ffill' (forward fill) or 'bfill' (backward fill)"],
["axis","Axis to fill on; default axis=0"],
["inplace","Modify the calling object without producing a copy"],
["limit","For forward and backward filling, maximum number of consecutive periods to fill"]
]

render_book_table(
    "Table 7-2. fillna Function Arguments",
    columns,
    rows
)

Argument,Description
value,Scalar value or dict-like object to use to fill missing values
method,Interpolation method; typically 'ffill' (forward fill) or 'bfill' (backward fill)
axis,Axis to fill on; default axis=0
inplace,Modify the calling object without producing a copy
limit,"For forward and backward filling, maximum number of consecutive periods to fill"


---

## 7.2 Data Transformation

So far in this chapter we’ve been concerned with rearranging data. Filtering, cleaning,
and other transformations are another class of important operations.

### Removing Duplicates

Duplicate rows may be found in a DataFrame for any number of reasons. Here is an
example:

In [48]:
data = pd.DataFrame({'k1': ['one', 'two'] * 3 + ['two'],'k2': [1, 1, 2, 3, 3, 4, 4]})

In [49]:
data

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4
6,two,4


The DataFrame method `duplicated` returns a boolean Series indicating whether each
row is a duplicate (has been observed in a previous row) or not:

In [51]:
data.duplicated()

0    False
1    False
2    False
3    False
4    False
5    False
6     True
dtype: bool

Relatedly, `drop_duplicates returns` a DataFrame where the duplicated array is
False:

In [52]:
data.drop_duplicates()

,k1,k2
0,one,1
1,two,1
2,one,2
3,two,3
4,one,3
5,two,4


Both of these methods by default consider all of the columns; alternatively, you can
specify any subset of them to detect duplicates. Suppose we had an additional column
of values and wanted to filter duplicates only based on the `'k1'` column:

In [53]:
data['v1'] = range(7)

In [54]:
data

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
5,two,4,5
6,two,4,6


In [55]:
data.drop_duplicates(['k1'])

,k1,k2,v1
0,one,1,0
1,two,1,1


`duplicated` and `drop_duplicates` by default keep the first observed value combination.
Passing `keep='last'` will return the last one:

In [57]:
data.drop_duplicates(['k1', 'k2'], keep='last')

,k1,k2,v1
0,one,1,0
1,two,1,1
2,one,2,2
3,two,3,3
4,one,3,4
6,two,4,6


---

### Transforming Data Using a Function or Mapping

For many datasets, you may wish to perform some transformation based on the values
in an array, Series, or column in a DataFrame. Consider the following hypothetical
data collected about various kinds of meat:

In [58]:
data = pd.DataFrame({'food': ['bacon', 'pulled pork', 'bacon','Pastrami', 'corned beef', 'Bacon','pastrami', 'honey ham', 'nova lox'],'ounces': [4, 3, 12, 6, 7.5, 8, 3, 5, 6]})

In [59]:
data

,food,ounces
0,bacon,4.0
1,pulled pork,3.0
2,bacon,12.0
3,Pastrami,6.0
4,corned beef,7.5
5,Bacon,8.0
6,pastrami,3.0
7,honey ham,5.0
8,nova lox,6.0


Suppose you wanted to add a column indicating the type of animal that each food
came from. Let’s write down a mapping of each distinct meat type to the kind of
animal:

In [60]:
meat_to_animal = {
'bacon': 'pig',
'pulled pork': 'pig',
'pastrami': 'cow',
'corned beef': 'cow',
'honey ham': 'pig',
'nova lox': 'salmon'
}

The `map` method on a Series accepts a function or dict-like object containing a mapping,
but here we have a small problem in that some of the meats are capitalized and
others are not. Thus, we need to convert each value to lowercase using the `str.lower`
Series method:

In [61]:
lowercased = data['food'].str.lower()

In [62]:
lowercased

0          bacon
1    pulled pork
2          bacon
3       pastrami
4    corned beef
5          bacon
6       pastrami
7      honey ham
8       nova lox
Name: food, dtype: object

In [63]:
data['animal'] = lowercased.map(meat_to_animal)

In [64]:
data

,food,ounces,animal
0,bacon,4.0,pig
1,pulled pork,3.0,pig
2,bacon,12.0,pig
3,Pastrami,6.0,cow
4,corned beef,7.5,cow
5,Bacon,8.0,pig
6,pastrami,3.0,cow
7,honey ham,5.0,pig
8,nova lox,6.0,salmon


We could also have passed a function that does all the work:

In [65]:
data['animal2'] = data['food'].map(lambda x: meat_to_animal[x.lower()])

In [66]:
data

,food,ounces,animal,animal2
0,bacon,4.0,pig,pig
1,pulled pork,3.0,pig,pig
2,bacon,12.0,pig,pig
3,Pastrami,6.0,cow,cow
4,corned beef,7.5,cow,cow
5,Bacon,8.0,pig,pig
6,pastrami,3.0,cow,cow
7,honey ham,5.0,pig,pig
8,nova lox,6.0,salmon,salmon


Using `map` is a convenient way to perform element-wise transformations and other
data cleaning–related operations.

---

### Replacing Values

Filling in missing data with the `fillna` method is a special case of more general value
replacement. As you’ve already seen, `map` can be used to modify a subset of values in
an object but `replace` provides a simpler and more flexible way to do so. Let’s consider
this Series:

In [67]:
data = pd.Series([1., -999., 2., -999., -1000., 3.])

In [68]:
data

0       1.0
1    -999.0
2       2.0
3    -999.0
4   -1000.0
5       3.0
dtype: float64

The `-999` values might be sentinel values for missing data. To replace these with NA
values that pandas understands, we can use replace, producing a new Series (unless
you pass `inplace=True`):

In [69]:
data.replace(-999, np.nan)

0       1.0
1       NaN
2       2.0
3       NaN
4   -1000.0
5       3.0
dtype: float64

If you want to replace multiple values at once, you instead pass a list and then the
substitute value:

In [70]:
data.replace([-999, -1000], np.nan)

0    1.0
1    NaN
2    2.0
3    NaN
4    NaN
5    3.0
dtype: float64

To use a different replacement for each value, pass a list of substitutes:

In [71]:
data.replace([-999, -1000], [np.nan, 0])

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

The argument passed can also be a dict:

In [72]:
data.replace({-999: np.nan, -1000: 0})

0    1.0
1    NaN
2    2.0
3    NaN
4    0.0
5    3.0
dtype: float64

> The data.replace method is distinct from data.str.replace,
which performs string substitution element-wise. We look at these
string methods on Series later in the chapter.

---

### Renaming Axis Indexes

Like values in a Series, axis labels can be similarly transformed by a function or mapping
of some form to produce new, differently labeled objects. You can also modify
the axes in-place without creating a new data structure. Here’s a simple example:

In [73]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),index=['Ohio', 'Colorado', 'New York'],columns=['one', 'two', 'three', 'four'])

In [74]:
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
New York,8,9,10,11


Like a Series, the axis indexes have a `map` method:

In [75]:
transform = lambda x: x[:4].upper()

In [76]:
data.index

Index(['Ohio', 'Colorado', 'New York'], dtype='object')

In [77]:
data.index.map(transform)

Index(['OHIO', 'COLO', 'NEW '], dtype='object')

You can assign to `index`, modifying the DataFrame in-place:

In [78]:
data.index = data.index.map(transform)

In [79]:
data

,one,two,three,four
OHIO,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


If you want to create a transformed version of a dataset without modifying the original,
a useful method is `rename`:

In [81]:
data.rename(index=str.title, columns=str.upper)

,ONE,TWO,THREE,FOUR
Ohio,0,1,2,3
Colo,4,5,6,7
New,8,9,10,11


Notably, `rename` can be used in conjunction with a dict-like object providing new values
for a subset of the axis labels

In [83]:
data.rename(index={'OHIO': 'INDIANA'},columns={'three': 'peekaboo'})

,one,two,peekaboo,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


`rename` saves you from the chore of copying the DataFrame manually and assigning
to its index and columns attributes. Should you wish to modify a dataset in-place,
pass `inplace=True`:

In [84]:
data.rename(index={'OHIO': 'INDIANA'}, inplace=True)

In [85]:
data

,one,two,three,four
INDIANA,0,1,2,3
COLO,4,5,6,7
NEW,8,9,10,11


---

### Discretization and Binning

Continuous data is often discretized or otherwise separated into “bins” for analysis.
Suppose you have data about a group of people in a study, and you want to group
them into discrete age buckets:

In [86]:
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]

Let’s divide these into bins of 18 to 25, 26 to 35, 36 to 60, and finally 61 and older. To
do so, you have to use cut, a function in pandas:

In [87]:
bins = [18, 25, 35, 60, 100]

In [88]:
cats = pd.cut(ages, bins)

In [89]:
cats

[(18, 25], (18, 25], (18, 25], (25, 35], (18, 25], ..., (25, 35], (60, 100], (35, 60], (35, 60], (25, 35]]
Length: 12
Categories (4, interval[int64, right]): [(18, 25] < (25, 35] < (35, 60] < (60, 100]]

In [90]:
type(cats[0])

pandas._libs.interval.Interval

The object pandas returns is a special Categorical object. The output you see
describes the bins computed by `pandas.cut`. You can treat it like an array of strings
indicating the `bin` name; internally it contains a `categories` array specifying the distinct
category names along with a labeling for the `ages` data in the `codes` attribute:

In [92]:
cats.codes

array([0, 0, 0, 1, 0, 0, 2, 1, 3, 2, 2, 1], dtype=int8)

In [93]:
cats.categories

IntervalIndex([(18, 25], (25, 35], (35, 60], (60, 100]], dtype='interval[int64, right]')

In [94]:
pd.value_counts(cats)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_39416\1485279302.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(cats)


(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

In [95]:
cats.value_counts() 

(18, 25]     5
(25, 35]     3
(35, 60]     3
(60, 100]    1
Name: count, dtype: int64

Note that `pd.value_counts(cats)` are the `bin` counts for the result of `pandas.cut`.

Consistent with mathematical notation for intervals, a parenthesis means that the side
is open, while the square bracket means it is *closed (inclusive)*. You can change which
side is closed by passing `right=False`:

In [97]:
pd.cut(ages, [18, 26, 36, 61, 100], right=False)

[[18, 26), [18, 26), [18, 26), [26, 36), [18, 26), ..., [26, 36), [61, 100), [36, 61), [36, 61), [26, 36)]
Length: 12
Categories (4, interval[int64, left]): [[18, 26) < [26, 36) < [36, 61) < [61, 100)]

You can also pass your own bin names by passing a list or array to the `labels` option:

In [98]:
group_names = ['Youth', 'YoungAdult', 'MiddleAged', 'Senior']

In [99]:
pd.cut(ages, bins, labels=group_names)

['Youth', 'Youth', 'Youth', 'YoungAdult', 'Youth', ..., 'YoungAdult', 'Senior', 'MiddleAged', 'MiddleAged', 'YoungAdult']
Length: 12
Categories (4, object): ['Youth' < 'YoungAdult' < 'MiddleAged' < 'Senior']

If you pass an integer number of bins to `cut` instead of explicit bin edges, it will compute
equal-length bins based on the minimum and maximum values in the data.
Consider the case of some uniformly distributed data chopped into fourths:

In [101]:
data = np.random.rand(20)

In [103]:
cats=pd.cut(data, 4, precision=2)

In [104]:
cats.value_counts()

(0.15, 0.34]    7
(0.34, 0.53]    1
(0.53, 0.72]    6
(0.72, 0.91]    6
Name: count, dtype: int64

The `precision=2` option limits the decimal precision to two digits.

A closely related function, `qcut`, bins the data based on sample quantiles. Depending
on the distribution of the data, using `cut` will not usually result in each bin having the
same number of data points. Since `qcut` uses sample quantiles instead, by definition

In [108]:
data = np.random.randn(1000) # Normally distributed

In [109]:
cats = pd.qcut(data, 4) # Cut into quartiles

In [110]:
cats

[(-3.3649999999999998, -0.688], (-3.3649999999999998, -0.688], (0.729, 3.073], (-0.688, -0.0128], (-3.3649999999999998, -0.688], ..., (-0.0128, 0.729], (-0.688, -0.0128], (-0.688, -0.0128], (0.729, 3.073], (0.729, 3.073]]
Length: 1000
Categories (4, interval[float64, right]): [(-3.3649999999999998, -0.688] < (-0.688, -0.0128] < (-0.0128, 0.729] < (0.729, 3.073]]

In [111]:
pd.value_counts(cats)

C:\Users\212364780.HCAD\AppData\Local\Temp\ipykernel_39416\1485279302.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  pd.value_counts(cats)


(-3.3649999999999998, -0.688]    250
(-0.688, -0.0128]                250
(-0.0128, 0.729]                 250
(0.729, 3.073]                   250
Name: count, dtype: int64

In [113]:
cats.value_counts()

(-3.3649999999999998, -0.688]    250
(-0.688, -0.0128]                250
(-0.0128, 0.729]                 250
(0.729, 3.073]                   250
Name: count, dtype: int64

Similar to `cut` you can pass your own quantiles (numbers between 0 and 1, inclusive):



In [118]:
cats=pd.qcut(data, [0, 0.1, 0.5, 0.9, 1.])

In [119]:
cats.value_counts()

(-3.3649999999999998, -1.267]    100
(-1.267, -0.0128]                400
(-0.0128, 1.369]                 400
(1.369, 3.073]                   100
Name: count, dtype: int64

We’ll return to `cut` and `qcut` later in the chapter during our discussion of aggregation
and group operations, as these discretization functions are especially useful for quantile
and group analysis.

---

### Detecting and Filtering Outliers

Filtering or transforming outliers is largely a matter of applying array operations.
Consider a DataFrame with some normally distributed data:

In [3]:
data = pd.DataFrame(np.random.randn(1000, 4))

In [4]:
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.017919,-0.032760,-0.023303,0.023233
std,1.018148,1.010236,0.970186,0.992753
min,-2.866907,-3.694504,-3.373598,-3.229335
25%,-0.654697,-0.689007,-0.662661,-0.622734
50%,-0.002502,-0.040712,-0.042080,0.035063
75%,0.693112,0.629698,0.592163,0.675016
max,2.987844,3.204884,2.955924,4.053345


Suppose you wanted to find values in one of the columns exceeding 3 in absolute
value:

In [5]:
col = data[2]

In [6]:
col

0      0.061747
1     -0.226541
2     -0.899644
3     -0.688062
4      0.464679
         ...   
995    0.162014
996    0.747371
997    0.672368
998   -0.958325
999    0.420825
Name: 2, Length: 1000, dtype: float64

In [7]:
col[np.abs(col) > 3]

302   -3.373598
885   -3.056888
Name: 2, dtype: float64

To select all rows having a value exceeding 3 or –3, you can use the `any` method on a
boolean DataFrame:

In [9]:
data[(np.abs(data) > 3).any(1)]

TypeError: DataFrame.any() takes 1 positional argument but 2 were given

this is a **very common pandas version issue** 👍

You are running code written for **older pandas**, but your environment has **newer pandas (≥ 2.x)**.

---

##### ✅ Why this error happens

Your code:

```python
data[(np.abs(data) > 3).any(1)]
```

Error:

```
TypeError: DataFrame.any() takes 1 positional argument but 2 were given
```

In **older pandas**, this was valid:

```python
.any(1)
```

because `1` was interpreted as:

```
axis = 1   (row-wise operation)
```

But in **new pandas**, `axis` must be passed as a **keyword argument**.

---

##### ✅ Correct Fix

Just change to:

```python
data[(np.abs(data) > 3).any(axis=1)]
```

---

##### ✅ What this code is doing (important concept)

Let’s understand like a **data science architect** 🙂

##### Step 1 — Generate data

```python
data = pd.DataFrame(np.random.randn(1000, 4))
```

Creates:

* 1000 rows
* 4 columns
* random normal values

---

##### Step 2 — Find extreme values

```python
np.abs(data) > 3
```

Creates a **boolean DataFrame**

Example:

| 0     | 1     | 2    | 3     |
| ----- | ----- | ---- | ----- |
| False | False | True | False |

Meaning → column 2 has extreme value (>3 or <-3)

---

##### Step 3 — `.any(axis=1)`

This is **VERY important in data cleaning / outlier detection**

```
axis = 1 → check row-wise
```

Meaning:

> “Keep rows where ANY column has an extreme value”

Result → boolean Series like:

```
0     False
1     True
2     False
```

---

##### Step 4 — Filter rows

```python
data[ ... ]
```

So final meaning:

> **Return all rows where at least one column has |value| > 3**

This is a classic:

⭐ **Outlier row detection pattern**

---

##### ✅ Very senior tip (production-quality style)

Better readable version:

```python
mask = (np.abs(data) > 3).any(axis=1)
outliers = data[mask]
```

This is:

* Debuggable
* Maintainable
* Preferred in pipelines

---

##### ✅ Ultra important pandas evolution insight

In modern pandas:

| Old Style | New Recommended Style |
| --------- | --------------------- |
| `.any(1)` | `.any(axis=1)`        |
| `.sum(0)` | `.sum(axis=0)`        |

Because pandas is becoming:

> **Strict → explicit → less ambiguous**

---

If you want, I can next show you something very powerful:

👉 **How to remove outliers using this pattern (z-score filtering / winsorization / IQR filtering)**
This is one of the most important real-world data cleaning skills.


data[(np.abs(data) > 3).any(axis=1)]

---

Values can be set based on these criteria. Here is code to cap values outside the interval
–3 to 3:

In [12]:
data[np.abs(data) > 3] = np.sign(data) * 3

In [14]:
data

,0,1,2,3
0,0.320600,0.206439,0.061747,1.547352
1,-0.381095,-0.903357,-0.226541,-0.868312
2,0.332688,-1.027716,-0.899644,-0.347976
3,0.680358,0.895297,-0.688062,1.099463
4,1.656474,-0.077179,0.464679,-0.232822
...,...,...,...,...
995,-1.115414,-0.881796,0.162014,-0.325367
996,-1.942350,0.244551,0.747371,-0.898751
997,-0.382582,1.534099,0.672368,1.500463
998,0.675474,0.429473,-0.958325,-0.347200


In [15]:
data.describe()

,0,1,2,3
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.017919,-0.032619,-0.022873,0.022315
std,1.018148,1.006175,0.968790,0.988007
min,-2.866907,-3.000000,-3.000000,-3.000000
25%,-0.654697,-0.689007,-0.662661,-0.622734
50%,-0.002502,-0.040712,-0.042080,0.035063
75%,0.693112,0.629698,0.592163,0.675016
max,2.987844,3.000000,2.955924,3.000000


In [16]:
np.sign(data) * 3

,0,1,2,3
0,3.0,3.0,3.0,3.0
1,-3.0,-3.0,-3.0,-3.0
2,3.0,-3.0,-3.0,-3.0
3,3.0,3.0,-3.0,3.0
4,3.0,-3.0,3.0,-3.0
...,...,...,...,...
995,-3.0,-3.0,3.0,-3.0
996,-3.0,3.0,3.0,-3.0
997,-3.0,3.0,3.0,3.0
998,3.0,3.0,-3.0,-3.0


The statement `np.sign(data)` produces 1 and –1 values based on whether the values
in data are positive or negative:

The statement : `data[np.abs(data) > 3]`, gets assigned values only for the values where absolute value is > 3 

This is a **very important NumPy + pandas broadcasting concept** and many people use it without deeply understanding what is happening.

Your statement:

```python
data[np.abs(data) > 3] = np.sign(data) * 3
```

Let’s break this **very carefully like a data science architect.**

---

##### ✅ What this statement is trying to do

It is performing **outlier capping (clipping).**

Meaning:

> Any value greater than +3 becomes +3
> Any value less than −3 becomes −3

Values inside range **remain unchanged.**

This technique is called:

⭐ **Winsorization / Capping**

---

##### ✅ Step-by-Step Execution

Assume a row looks like:

```
[ 1.2 , 4.5 , -3.8 , 0.5 ]
```

---

##### ⭐ Step 1 — Create Boolean Mask

```python
np.abs(data) > 3
```

Gives:

```
[ False , True , True , False ]
```

Meaning:

* Only column 2 and 3 are outliers.

---

##### ⭐ Step 2 — Right side expression

```python
np.sign(data) * 3
```

Very important.

##### What does `np.sign()` do?

| Value    | sign |
| -------- | ---- |
| positive | +1   |
| negative | -1   |
| zero     | 0    |

So:

```
np.sign([1.2 , 4.5 , -3.8 , 0.5])
→ [1 , 1 , -1 , 1]
```

Multiply by 3:

```
[3 , 3 , -3 , 3]
```

---

##### ⭐ Step 3 — Masked Assignment (MOST IMPORTANT)

Now pandas applies assignment **ONLY where mask is True**

So:

```
Original:
[ 1.2 , 4.5 , -3.8 , 0.5 ]

Mask:
[ False , True , True , False ]

Replacement values:
[ 3 , 3 , -3 , 3 ]
```

Final result:

```
[ 1.2 , 3 , -3 , 0.5 ]
```

---

##### ✅ VERY VERY IMPORTANT CONCEPT

##### 👉 Pandas does NOT replace entire row

It replaces **only individual cells where condition is True**

This is called:

⭐ **Boolean Masked Vectorized Assignment**

This is one of the most powerful patterns in pandas.

---

##### ✅ Why values become ONLY +3 or −3

Because:

```
np.sign(data)
```

returns **only −1 or +1**

Then multiplied by 3 →

```
−1 × 3 = −3  
+1 × 3 = +3
```

So:

* 10 becomes → +3
* 5 becomes → +3
* −100 becomes → −3
* −4 becomes → −3

---

##### ✅ Visual mental model (VERY helpful)

Think like this:

> “Where value exceeds limit → push it back to boundary in same direction”

Exactly like:

* speed limit enforcement
* temperature clipping
* sensor saturation

---

##### ✅ Senior Data Science Insight

This pattern is equivalent to:

```python
data = data.clip(-3, 3)
```

Which is:

⭐ Cleaner
⭐ Faster
⭐ More expressive

---

##### ✅ Even more production-grade version

```python
data = data.clip(lower=-3, upper=3)
```

---

##### 🚨 One subtle deep insight (very few people notice)

Your version:

```python
data[np.abs(data) > 3] = np.sign(data) * 3
```

works because:

* pandas aligns mask
* broadcasts RHS
* assigns cell-wise

But `clip()` is implemented in **C-optimized code**

So in large datasets:

👉 `clip()` is significantly faster.


### Permutation and Random Sampling

Permuting (randomly reordering) a Series or the rows in a DataFrame is easy to do
using the `numpy.random.permutation` function. Calling permutation with the length
of the axis you want to permute produces an array of integers indicating the new
ordering:

In [4]:
df = pd.DataFrame(np.arange(5 * 4).reshape((5, 4)))

In [5]:
sampler = np.random.permutation(5)

In [6]:
sampler

array([0, 1, 3, 2, 4], dtype=int32)

In [8]:
df

,0,1,2,3
0,0,1,2,3
1,4,5,6,7
2,8,9,10,11
3,12,13,14,15
4,16,17,18,19


That array can then be used in iloc-based indexing or the equivalent `take` function:

In [12]:
df.take(sampler)

,0,1,2,3
0,0,1,2,3
1,4,5,6,7
3,12,13,14,15
2,8,9,10,11
4,16,17,18,19


To select a random subset without replacement, you can use the `sample` method on
Series and DataFrame:

In [13]:
df.sample(n=3)

,0,1,2,3
2,8,9,10,11
4,16,17,18,19
0,0,1,2,3


To generate a sample with replacement (to allow repeat choices), pass `replace=True`
to sample:

In [15]:
choices = pd.Series([5, 7, -1, 6, 4])

In [16]:
draws = choices.sample(n=10, replace=True)

In [17]:
draws

1    7
2   -1
1    7
1    7
3    6
3    6
1    7
3    6
4    4
4    4
dtype: int64

### Computing Indicator/Dummy Variables

Another type of transformation for statistical modeling or machine learning applications
is converting a categorical variable into a “dummy” or “indicator” matrix. If a
column in a DataFrame has k distinct values, you would derive a matrix or Data‐
Frame with k columns containing all 1s and 0s. pandas has a `get_dummies` function
for doing this, though devising one yourself is not difficult. Let’s return to an earlier
example DataFrame:

In [18]:
df = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'b'],'data1': range(6)})

In [19]:
df

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,b,5


In [20]:
pd.get_dummies(df['key'])

,a,b,c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


you can see that the column 'a' has value only for rows where colum key has value'a'

#### How to do this with out the dummies function. 

##### ⭐ Step 0 — Your Data

```python
df['key']
```

```
0    b
1    b
2    a
3    c
4    a
5    b
```

---

##### ⭐ Step 1 — `keys = df['key'].unique()`

This returns **distinct category values**.

```python
keys = ['b', 'a', 'c']
```

Important points:

* Order = order of first appearance
* Not sorted
* Type = NumPy array

Shape:

```
(3,)
```

---

##### ⭐ Step 2 — `df['key'].values`

Convert pandas Series → **NumPy array**

```
['b','b','a','c','a','b']
```

Shape:

```
(6,)
```

This is a **1-D flat vector**

It is neither a row nor a column.

---

##### ⭐ Step 3 — Understanding `[:, None]`

This is **NumPy indexing syntax**.

It means:

> Select all elements and add a NEW axis.

Breakdown:

* `:` → take all elements
* `None` → insert a new dimension

Equivalent:

```python
arr[:, np.newaxis]
```

---

##### ⭐ Shape Change

Before:

```
ndim = 1  
shape = (6,)
```

After:

```
ndim = 2  
shape = (6,1)
```

We converted:

```
Flat vector → Column vector
```

---

##### ⭐ Visual Understanding

Before:

```
[b  b  a  c  a  b]
```

After:

```
[[b]
 [b]
 [a]
 [c]
 [a]
 [b]]
```

Now we have:

* 6 rows
* 1 column

---

##### ⭐ Very Important — No Memory Copy

This is NOT an expensive operation.

NumPy only changes:

* shape metadata
* strides

So vectorized code stays **very fast**.

---

##### ⭐ Why Do We Need Column Vector?

Because of **broadcasting rules**.

Later we compare:

```
(6,1)   with   (3,)
```

NumPy expands them to:

```
(6,3)
```

This allows comparison of **every row with every category**.

---

##### ⭐ Step 4 — Broadcasting Comparison

```python
df['key'].values[:, None] == keys
```

NumPy interprets this like:

```
[['b','b','b'],
 ['b','b','b'],
 ['a','a','a'],
 ['c','c','c'],
 ['a','a','a'],
 ['b','b','b']]
==
['b','a','c']
```

---

##### ⭐ Boolean Matrix Result

```
[[ True False False ]
 [ True False False ]
 [ False True False ]
 [ False False True ]
 [ False True False ]
 [ True False False ]]
```

Shape:

```
(6,3)
```

This is already the **one-hot structure**.

---

##### ⭐ Step 5 — `.astype(int)`

Convert boolean → numeric

```
True → 1  
False → 0
```

Matrix becomes:

```
[[1 0 0]
 [1 0 0]
 [0 1 0]
 [0 0 1]
 [0 1 0]
 [1 0 0]]
```

---

##### ⭐ Step 6 — Wrap into DataFrame

```python
one_hot = pd.DataFrame(matrix, columns=keys)
```

Final Output:

```
   b  a  c
0  1  0  0
1  1  0  0
2  0  1  0
3  0  0  1
4  0  1  0
5  1  0  0
```

---

##### ⭐ Equivalent Ways to Add New Axis

All give same result:

```python
arr[:, None]
arr.reshape(-1,1)
np.expand_dims(arr, axis=1)
```

---

##### ⭐ Architect Mental Model (Remember Forever)

`None` means:

> “Insert a dimension here.”

Examples:

```
arr.shape = (6,)
```

| Operation      | Result Shape | Meaning       |
| -------------- | ------------ | ------------- |
| `arr[None, :]` | (1,6)        | row vector    |
| `arr[:, None]` | (6,1)        | column vector |

---

##### ⭐ Deep Mental Model of One-Hot Encoding

This line is basically saying:

> Compare every row category with every possible category
> and mark 1 where equal.

This is called:

**Vectorized outer comparison**

---

##### ⭐ Why This Pattern is Extremely Powerful

✅ No Python loops
✅ Pure NumPy C-level execution
✅ Scales to millions of rows
✅ Foundation of ML feature engineering
✅ Used internally by sklearn / deep learning tensor ops

---

##### ⭐ Ultra Powerful Intuition

Think:

```
Take one column  
Turn it sideways  
Compare with all labels  
→ You get a one-hot grid
```


----

In some cases, you may want to add a prefix to the columns in the indicator Data‐
Frame, which can then be merged with the other data. `get_dummies` has a `prefix` argument
for doing this:

In [27]:
dummies = pd.get_dummies(df['key'], prefix='key')

In [28]:
dummies

,key_a,key_b,key_c
0,False,True,False
1,False,True,False
2,True,False,False
3,False,False,True
4,True,False,False
5,False,True,False


In [29]:
df[['data1']].join(dummies)  # DF with dumnmmies

,data1,key_a,key_b,key_c
0,0,False,True,False
1,1,False,True,False
2,2,True,False,False
3,3,False,False,True
4,4,True,False,False
5,5,False,True,False


----

If a row in a DataFrame belongs to multiple categories, things are a bit more complicated.
Let’s look at the `MovieLens 1M` dataset, which is investigated in more detail in
Chapter 14:

Downloaded the data set from : https://grouplens.org/datasets/movielens/

In [31]:
mnames = ['movie_id', 'title', 'genres']

In [34]:
movies = pd.read_csv('examples/ml-32m/ml-32m/movies.csv', sep=',',header=1, names=mnames)

In [35]:
movies

,movie_id,title,genres
0,2,Jumanji (1995),Adventure|Children|Fantasy
1,3,Grumpier Old Men (1995),Comedy|Romance
2,4,Waiting to Exhale (1995),Comedy|Drama|Romance
3,5,Father of the Bride Part II (1995),Comedy
4,6,Heat (1995),Action|Crime|Thriller
...,...,...,...
87579,292731,The Monroy Affaire (2022),Drama
87580,292737,Shelter in Solitude (2023),Comedy|Drama
87581,292753,Orca (2023),Drama
87582,292755,The Angry Breed (1968),Drama


Adding indicator variables for each genre requires a little bit of wrangling. First, we
extract the list of unique genres in the dataset:

In [37]:
all_genres = []

In [40]:
all_genres = movies['genres'].str.split('|').explode()

In [41]:
genres = pd.unique(all_genres)

In [42]:
all_genres

0          Adventure
0           Children
0            Fantasy
1             Comedy
1            Romance
            ...     
87581          Drama
87582          Drama
87583         Action
87583      Adventure
87583    Documentary
Name: genres, Length: 154165, dtype: object

In [43]:
genres

array(['Adventure', 'Children', 'Fantasy', 'Comedy', 'Romance', 'Drama',
       'Action', 'Crime', 'Thriller', 'Horror', 'Animation', 'Mystery',
       'Sci-Fi', 'IMAX', 'Documentary', 'War', 'Musical', 'Western',
       'Film-Noir', '(no genres listed)'], dtype=object)

##### ✅ Statement

```python
all_genres = movies['genres'].str.split('|').explode()
```

This line performs **two major data transformations**:

1. Converts a **multi-value string → list**
2. Converts a **list → multiple rows**

This pattern is extremely common in **real data science pipelines** (especially recommender systems, NLP datasets, logs, tagging systems, etc.).

---

##### ✅ Step 1 — What `movies['genres']` Looks Like

In the MovieLens dataset, the column contains **pipe-separated genre strings**.

Example:

```python
0    Adventure|Animation|Children|Comedy|Fantasy
1    Comedy|Romance
2    Action|Thriller
```

So the datatype is:

```
Pandas Series of strings
```

Each row represents **one movie with possibly multiple genres**.

---

##### ✅ Step 2 — `.str.split('|')`

This uses **Pandas vectorized string operations**.

```python
movies['genres'].str.split('|')
```

Now each string becomes a **Python list**.

Result:

```python
0    ['Adventure','Animation','Children','Comedy','Fantasy']
1    ['Comedy','Romance']
2    ['Action','Thriller']
```

Datatype becomes:

```
Series of lists
```

So we moved from:

```
Scalar string  →  list object
```

This is the first normalization step.

---

##### ✅ Step 3 — `.explode()`

This is the **key operation** ⭐

`explode()` means:

> If a row contains a list → create multiple rows (one per list element)

Example:

Before explode:

| index | genres_list                          |
| ----- | ------------------------------------ |
| 0     | ['Adventure','Animation','Children'] |
| 1     | ['Comedy','Romance']                 |

After explode:

| index | genre     |
| ----- | --------- |
| 0     | Adventure |
| 0     | Animation |
| 0     | Children  |
| 1     | Comedy    |
| 1     | Romance   |

Important observation:

👉 **Index repeats**

This is intentional and extremely useful later.

---

##### ✅ Final Output Stored in `all_genres`

```python
0 Adventure
0 Animation
0 Children
1 Comedy
1 Romance
2 Action
2 Thriller
...
```

Datatype:

```
Pandas Series
```

Not:

* Python list
* DataFrame
* NumPy array

---

##### ✅ Why This is Very Powerful (Architect Insight)

Movie genres form a **multi-label categorical feature**.

This transformation enables downstream analytics.

---

##### ⭐ 1. Count Genre Popularity

```python
all_genres.value_counts()
```

Now you know:

* Most common genres
* Dataset bias
* Useful prior for recommender systems

---

##### ⭐ 2. One-Hot Encode Genres

```python
pd.get_dummies(all_genres)
```

This helps build:

```
Feature matrices for ML / recommendation models
```

---

##### ⭐ 3. Join Back to Movie IDs

Because explode **preserves index**, you can later do:

```python
all_genres.groupby(level=0)
```

This allows reconstruction of:

```
Movie → Genre relationships
```

This index-preservation behavior is a **very deliberate Pandas design decision**.

---

##### ✅ Equivalent Slow Python Loop (Your Original Logic)

```python
all_genres = []

for x in movies.genres:
    all_genres.extend(x.split('|'))
```

This produces:

```
Plain Python list
```

Problems:

* Loses mapping to movie rows
* Slow for large datasets
* Not vectorized
* Hard to aggregate or join

The explode solution solves all of these.

---

##### ⭐ Very Important Mental Model

Think of `explode()` as:

```
Wide row containing list
            ↓
Tall table with repeated rows
```

Modern data science workflows strongly prefer **“tall / long format data.”**

Because:

* `groupby` becomes natural
* joins become clean
* aggregations become scalable
* feature engineering becomes easier

---

##### 📌 Context Alignment with Your Project

Since this project is meant for structured data-science learning and architectural understanding , mastering patterns like **split → explode → aggregate → encode** is foundational for:

* recommender systems
* categorical feature engineering
* log/event analytics
* NLP token pipelines

---

##### 🚀 If you want — next we can go deeper into something very valuable:

* Build **Movie × Genre One-Hot Matrix (industry recommender input)**
* Visualize explode using a toy dataframe step-by-step
* Merge this with `ratings.csv` to start a **real recommendation model pipeline**

What would you like next?


----

One way to construct the indicator DataFrame is to start with a DataFrame of all
zeros:

In [45]:
zero_matrix = np.zeros((len(movies), len(genres)))

In [46]:
zero_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(87584, 20))

In [47]:
dummies = pd.DataFrame(zero_matrix, columns=genres)

In [49]:
dummies

,Adventure,Children,Fantasy,Comedy,Romance,Drama,Action,Crime,Thriller,Horror,Animation,Mystery,Sci-Fi,IMAX,Documentary,War,Musical,Western,Film-Noir,(no genres listed)
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87579,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
87580,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
87581,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
87582,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [52]:
genre_ohe = movies['genres'].str.get_dummies(sep='|')

movies_ohe = (
    movies
    .drop(columns='genres')
    .join(genre_ohe)
)

In [53]:
genre_ohe

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87579,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
87580,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0
87581,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
87582,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


In [54]:
movies_ohe

,movie_id,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,2,Jumanji (1995),0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,3,Grumpier Old Men (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
2,4,Waiting to Exhale (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,5,Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,6,Heat (1995),0,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87579,292731,The Monroy Affaire (2022),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
87580,292737,Shelter in Solitude (2023),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
87581,292753,Orca (2023),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
87582,292755,The Angry Breed (1968),0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


---- 

##### ✅ Goal

Understand:

1️⃣ What **`movies['genres'].str` actually holds**
2️⃣ How to use it to **One-Hot Encode genres without looping**

These two concepts are deeply connected in real Data Science preprocessing.

---

##### ✅ Step 1 — What is `movies['genres']`

```python
type(movies['genres'])
```

Result:

```
pandas.core.series.Series
```

Example values:

```
0    Adventure|Animation|Comedy
1    Comedy|Romance
2    Action|Thriller
```

So this column is:

> A **Series of strings** (multi-label categorical stored as text)

---

##### ✅ Step 2 — What Does `.str` Hold

```python
movies['genres'].str
```

This holds:

```
pandas.core.strings.accessor.StringMethods
```

Important:

❗ It is **NOT the data**
❗ It is **NOT another Series**

It is a **vectorized string-operation accessor (toolkit).**

Think:

```
Series
   ↓ attach text toolkit
StringMethods accessor (.str)
```

---

##### ⭐ Very Important Mental Model

Pandas provides special accessors:

| Accessor | Purpose                      |
| -------- | ---------------------------- |
| `.str`   | vectorized string operations |
| `.dt`    | datetime operations          |
| `.cat`   | categorical operations       |

So `.str` means:

> “Treat this entire column as text and give me text processing functions.”

---

##### ✅ Example Methods Available

After writing:

```python
movies['genres'].str.
```

You can use:

* `split()`
* `contains()`
* `replace()`
* `len()`
* `startswith()`
* `get_dummies()`

All work **row-wise automatically (vectorized).**

Example:

```python
movies['genres'].str.split('|')
```

Means:

> Apply Python string `.split('|')`
> to every row — **without writing loops.**

---

##### 🚀 Architect Insight — Why `.str` Exists

Loop version:

```python
for x in movies['genres']:
    x.split('|')
```

Problems:

* Slow
* Not scalable
* Breaks pipeline composability

Vectorized `.str` methods are:

* C-optimized
* memory efficient
* production-grade
* pipeline friendly

---

##### ✅ One-Hot Encoding Using `.str` (No Loop)

Best industry solution:

```python
movies_ohe = movies.join(
    movies['genres'].str.get_dummies(sep='|')
)
```

---

##### ✅ What `str.get_dummies()` Does

It directly converts:

```
Adventure|Animation|Comedy
```

into:

| Action | Adventure | Animation | Comedy |
| ------ | --------- | --------- | ------ |
| 0      | 1         | 1         | 1      |

Internally Pandas performs:

```
split → reshape → pivot → fill zeros
```

All vectorized.

---

##### ✅ Cleaner Production Version

Usually we drop original text column:

```python
genre_ohe = movies['genres'].str.get_dummies(sep='|')

movies_ohe = (
    movies
    .drop(columns='genres')
    .join(genre_ohe)
)
```

Now dataframe is **mostly numeric → ML ready.**

---

##### ⭐ Extremely Important Recommender System Step

Often we set:

```python
movies_ohe = movies_ohe.set_index('movie_id')
```

Now you have:

```
Movie Feature Matrix
```

Used for:

* Content-based filtering
* similarity computation
* hybrid recommender models

---

##### 🔥 Advanced Alternative Using Explode (Architect Trick)

If you already created:

```python
all_genres = movies['genres'].str.split('|').explode()
```

Then:

```python
genre_ohe = pd.get_dummies(all_genres).groupby(level=0).max()

movies_ohe = movies.join(genre_ohe)
```

This pipeline is useful when:

* preprocessing is complex
* multi-stage feature engineering required

---

##### ✅ Final Mental Model to Remember

```
Series of multi-label strings
        ↓  (.str accessor gives vectorized tools)
Dummy Encoding
        ↓
Feature Matrix
        ↓
ML / Recommender Input
```

---


A useful recipe for statistical applications is to combine `get_dummies with a discretization
function like `cut`:

In [56]:
np.random.seed(12345)

In [57]:
values = np.random.rand(10)

In [58]:
values

array([0.92961609, 0.31637555, 0.18391881, 0.20456028, 0.56772503,
       0.5955447 , 0.96451452, 0.6531771 , 0.74890664, 0.65356987])

In [59]:
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

In [63]:
pd.cut(values, bins)

[(0.8, 1.0], (0.2, 0.4], (0.0, 0.2], (0.2, 0.4], (0.4, 0.6], (0.4, 0.6], (0.8, 1.0], (0.6, 0.8], (0.6, 0.8], (0.6, 0.8]]
Categories (5, interval[float64, right]): [(0.0, 0.2] < (0.2, 0.4] < (0.4, 0.6] < (0.6, 0.8] < (0.8, 1.0]]

In [60]:
pd.get_dummies(pd.cut(values, bins))

,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,False,False,False,False,True
1,False,True,False,False,False
2,True,False,False,False,False
3,False,True,False,False,False
4,False,False,True,False,False
5,False,False,True,False,False
6,False,False,False,False,True
7,False,False,False,True,False
8,False,False,False,True,False
9,False,False,False,True,False


In [68]:
pd.Series(values) 

0    0.929616
1    0.316376
2    0.183919
3    0.204560
4    0.567725
5    0.595545
6    0.964515
7    0.653177
8    0.748907
9    0.653570
dtype: float64

In [69]:
## CVall in one code. 
np.random.seed(12345)
values = np.random.rand(10)
bins = [0, 0.2, 0.4, 0.6, 0.8, 1]

df = pd.DataFrame({'value': values})

cats = pd.cut(df['value'], bins)
dummies = pd.get_dummies(cats)

result = pd.concat([df, dummies], axis=1)

In [70]:
result

,value,"(0.0, 0.2]","(0.2, 0.4]","(0.4, 0.6]","(0.6, 0.8]","(0.8, 1.0]"
0,0.929616,False,False,False,False,True
1,0.316376,False,True,False,False,False
2,0.183919,True,False,False,False,False
3,0.204560,False,True,False,False,False
4,0.567725,False,False,True,False,False
5,0.595545,False,False,True,False,False
6,0.964515,False,False,False,False,True
7,0.653177,False,False,False,True,False
8,0.748907,False,False,False,True,False
9,0.653570,False,False,False,True,False


## 7.3 String Manipulation

Python has long been a popular raw data manipulation language in part due to its
ease of use for string and text processing. Most text operations are made simple with
the string object’s built-in methods. For more complex pattern matching and text
manipulations, regular expressions may be needed. pandas adds to the mix by enabling
you to apply string and regular expressions concisely on whole arrays of data,
additionally handling the annoyance of missing data.

### String Object Methods

In many string munging and scripting applications, built-in string methods are sufficient.
As an example, a comma-separated string can be broken into pieces with
`split` :

In [72]:
val = 'a,b, guido'

In [73]:
val.split(',')

['a', 'b', ' guido']

`split` is often combined with `strip` to trim whitespace (including line breaks):

In [74]:
pieces = [x.strip() for x in val.split(',')]

In [75]:
pieces

['a', 'b', 'guido']

In [76]:
'::'.join(pieces)

'a::b::guido'

Other methods are concerned with locating substrings. Using Python’s `in` keyword is
the best way to detect a substring, though `index` and `find`can also be used:



In [77]:
print( pieces)

['a', 'b', 'guido']


In [93]:
'guido' in val

True

In [94]:
val.index(',')

1

In [95]:
val.find(':')

-1

Note the difference between `find` and `index` is that `index raises an exception if the
string isn’t found (versus returning –1):

In [97]:
val.index(':')

ValueError: substring not found

Relatedly, `count` returns the number of occurrences of a particular substring:

In [100]:
val.count(',')

2

`replace` will substitute occurrences of one pattern for another. It is commonly used
to delete patterns, too, by passing an empty string:

In [101]:
val.replace(',', '::')

'a::b:: guido'

In [102]:
val.replace(',', '')

'ab guido'

See Table 7-3 for a listing of some of Python’s string methods.

In [103]:
columns = ["Argument", "Description"]

rows = [
["count","Return the number of non-overlapping occurrences of substring in the string"],
["endswith","Returns True if string ends with suffix"],
["startswith","Returns True if string starts with prefix"],
["join","Use string as delimiter for concatenating a sequence of other strings"],
["index","Return position of first character in substring; raises ValueError if not found"],
["find","Return position of first occurrence of substring; like index but returns −1 if not found"],
["rfind","Return position of last occurrence of substring; returns −1 if not found"],
["replace","Replace occurrences of string with another string"],
["strip / rstrip / lstrip","Trim whitespace including newlines"],
["split","Break string into list of substrings using delimiter"],
["lower","Convert alphabet characters to lowercase"],
["upper","Convert alphabet characters to uppercase"],
["casefold","Convert characters to lowercase in a locale-independent comparable form"],
["ljust / rjust","Left or right justify string and pad to minimum width"]
]

render_book_table(
    "Table 7-3. Python Built-in String Methods",
    columns,
    rows
)

Argument,Description
count,Return the number of non-overlapping occurrences of substring in the string
endswith,Returns True if string ends with suffix
startswith,Returns True if string starts with prefix
join,Use string as delimiter for concatenating a sequence of other strings
index,Return position of first character in substring; raises ValueError if not found
find,Return position of first occurrence of substring; like index but returns −1 if not found
rfind,Return position of last occurrence of substring; returns −1 if not found
replace,Replace occurrences of string with another string
strip / rstrip / lstrip,Trim whitespace including newlines
split,Break string into list of substrings using delimiter


#### Regular Expressions

Regular expressions provide a flexible way to search or match (often more complex)
string patterns in text. A single expression, commonly called a regex, is a string
formed according to the regular expression language. Python’s built-in `re` module is
responsible for applying regular expressions to strings; I’ll give a number of examples
of its use here.

> The art of writing regular expressions could be a chapter of its own
and thus is outside the book’s scope. There are many excellent tutorials
and references available on the internet and in other books.

The `**re**` module functions fall into three categories: pattern matching, substitution,
and splitting. Naturally these are all related; a regex describes a pattern to locate in the
text, which can then be used for many purposes. Let’s look at a simple example:

suppose we wanted to split a string with a variable number of whitespace characters
(tabs, spaces, and newlines). The regex describing one or more whitespace characters
is `\s+`:

In [104]:
import re
text = "foo bar\t baz \tqux"
re.split('\s+', text)

['foo', 'bar', 'baz', 'qux']

When you call `re.split('\s+', text)`, the regular expression is first compiled, and
then its `split` method is called on the passed text. You can compile the regex yourself
with `re.compile`, forming a reusable regex object:

In [106]:
regex = re.compile('\s+')

In [107]:
regex.split(text)

['foo', 'bar', 'baz', 'qux']

If, instead, you wanted to get a list of all patterns matching the regex, you can use the
`findall` method:

> Creating a regex object with `re.compile` is highly recommended if you intend to
apply the same expression to many strings; doing so will save CPU cycles.

----

`match` and `search` are closely related to findall. While `findall` returns all matches
in a string, `search` returns only the first match. More rigidly, `match` only matches at
the beginning of the string. As a less trivial example, let’s consider a block of text and
a regular expression capable of identifying most email addresses:

In [109]:
text = """Dave dave@google.com
Steve steve@gmail.com
Rob rob@gmail.com
Ryan ryan@yahoo.com
"""

In [110]:
pattern = r'[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,4}'

In [111]:
# re.IGNORECASE makes the regex case-insensitive
regex = re.compile(pattern, flags=re.IGNORECASE)

Using `findall` on the text produces a list of the email addresses:

In [113]:
regex.findall(text)

['dave@google.com', 'steve@gmail.com', 'rob@gmail.com', 'ryan@yahoo.com']

----

`search` returns a special `match` object for the first email address in the text. For the
preceding regex, the `match` object can only tell us the start and end position of the
pattern in the string:

In [114]:
m = regex.search(text)

In [115]:
m

<re.Match object; span=(5, 20), match='dave@google.com'>

In [116]:
text[m.start():m.end()]

'dave@google.com'

`regex.match` returns None, as it only will match if the pattern occurs at the start of the
string:

In [117]:
print(regex.match(text))

None


Relatedly, `sub` will return a new string with occurrences of the pattern replaced by the
a new string:


In [118]:
print(regex.sub('REDACTED', text))

Dave REDACTED
Steve REDACTED
Rob REDACTED
Ryan REDACTED



Suppose you wanted to find email addresses and simultaneously segment each
address into its three components: username, domain name, and domain suffix. To
do this, put parentheses around the parts of the pattern to segment:

In [119]:
pattern = r'([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})'

In [120]:
regex = re.compile(pattern, flags=re.IGNORECASE)

In [121]:
m = regex.match('wesm@bright.net')

In [122]:
m.groups()

('wesm', 'bright', 'net')

`findall` returns a list of tuples when the pattern has groups:

In [124]:
regex.findall(text)

[('dave', 'google', 'com'),
 ('steve', 'gmail', 'com'),
 ('rob', 'gmail', 'com'),
 ('ryan', 'yahoo', 'com')]

`sub` also has access to groups in each match using special symbols like \1 and \2. The
symbol \1 corresponds to the first matched group, \2 corresponds to the second, and
so forth:

In [126]:
print(regex.sub(r'Username: \1, Domain: \2, Suffix: \3', text))

Dave Username: dave, Domain: google, Suffix: com
Steve Username: steve, Domain: gmail, Suffix: com
Rob Username: rob, Domain: gmail, Suffix: com
Ryan Username: ryan, Domain: yahoo, Suffix: com



There is much more to regular expressions in Python, most of which is outside the
book’s scope. Table 7-4 provides a brief summary.

In [127]:
columns = ["Argument", "Description"]

rows = [
["findall","Return all non-overlapping matching patterns in a string as a list"],
["finditer","Like findall, but returns an iterator"],
["match","Match pattern at start of string and optionally segment pattern components into groups; returns match object or None"],
["search","Scan string for match anywhere in string; returns match object if found"],
["split","Break string into pieces at each occurrence of pattern"],
["sub / subn","Replace all (sub) or first n occurrences (subn) of pattern with replacement expression; use \\1, \\2, ... for group references"]
]

render_book_table(
    "Table 7-4. Regular Expression Methods",
    columns,
    rows
)

Argument,Description
findall,Return all non-overlapping matching patterns in a string as a list
finditer,"Like findall, but returns an iterator"
match,Match pattern at start of string and optionally segment pattern components into groups; returns match object or None
search,Scan string for match anywhere in string; returns match object if found
split,Break string into pieces at each occurrence of pattern
sub / subn,"Replace all (sub) or first n occurrences (subn) of pattern with replacement expression; use \1, \2, ... for group references"


---

### Vectorized String Functions in pandas

Cleaning up a messy dataset for analysis often requires a lot of string munging and
regularization. To complicate matters, a column containing strings will sometimes
have missing data:

In [129]:
data = {'Dave': 'dave@google.com', 'Steve': 'steve@gmail.com','Rob': 'rob@gmail.com', 'Wes': np.nan}

In [130]:
data = pd.Series(data)

In [131]:
data

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: object

In [132]:
data.isnull()

Dave     False
Steve    False
Rob      False
Wes       True
dtype: bool

You can apply string and regular expression methods can be applied (passing a
`lambda` or other function) to each value using `data.map`, but it will fail on the NA
(null) values. To cope with this, Series has array-oriented methods for string operations
that skip NA values. These are accessed through Series’s str attribute; for example,
we could check whether each email address has 'gmail' in it with str.contains:

In [134]:
data.str.contains('gmail')

Dave     False
Steve     True
Rob       True
Wes        NaN
dtype: object

Regular expressions can be used, too, along with any `re` options like 'IGNORECASE'

In [136]:
pattern

'([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\\.([A-Z]{2,4})'

In [137]:
data.str.findall(pattern, flags=re.IGNORECASE)

Dave     [(dave, google, com)]
Steve    [(steve, gmail, com)]
Rob        [(rob, gmail, com)]
Wes                        NaN
dtype: object

In [138]:
matches = data.str.match(pattern, flags=re.IGNORECASE)

In [139]:
matches

Dave     True
Steve    True
Rob      True
Wes       NaN
dtype: object

##### ✅ What is **Vectorized Element Retrieval** in Pandas?

In **pandas**, *vectorized element retrieval* means:

> Retrieving elements (values / groups / substrings) from **an entire Series (or DataFrame column) at once**,
> without writing Python loops — using optimized internal (NumPy-based / C-level) operations.

This is **fast, concise, and idiomatic pandas code**.

This style of working with whole arrays/columns together is a core idea in data science workflows. 

---

##### ✅ First — Understand Your Code Context

You have:

```python
data = {'Dave': 'dave@google.com', 
        'Steve': 'steve@gmail.com',
        'Rob': 'rob@gmail.com', 
        'Wes': np.nan}

pattern = '([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\\.([A-Z]{2,4})'

matches = data.str.match(pattern, flags=re.IGNORECASE)
matches.str.get(1)
```

Let’s fix one conceptual point first:

👉 `data` must be a **pandas Series**, not a Python dict.

Correct version:

```python
data = pd.Series({
    'Dave': 'dave@google.com',
    'Steve': 'steve@gmail.com',
    'Rob': 'rob@gmail.com',
    'Wes': np.nan
})
```

---

##### ✅ Step 1 — What does `str.match()` return?

```python
matches = data.str.match(pattern, flags=re.IGNORECASE)
```

Here:

* Pandas applies regex **to every element of the Series**
* This is **vectorized regex matching**

Conceptually:

| Name  | Email                                     | Regex Match |
| ----- | ----------------------------------------- | ----------- |
| Dave  | [dave@google.com](mailto:dave@google.com) | True        |
| Steve | [steve@gmail.com](mailto:steve@gmail.com) | True        |
| Rob   | [rob@gmail.com](mailto:rob@gmail.com)     | True        |
| Wes   | NaN                                       | NaN         |

So `matches` is:

```
Dave     True
Steve    True
Rob      True
Wes      NaN
dtype: object
```

---

##### ⚠️ Important

`str.match()` only tells **whether pattern matched**

It does **NOT return captured groups.**

To extract groups we should use:

```
str.extract()
```

---

##### ✅ Correct Flow for Vectorized Element Retrieval

```python
matches = data.str.extract(pattern, flags=re.IGNORECASE)
```

Now pandas returns:

| index | group 0 (username) | group 1 (domain) | group 2 (TLD) |
| ----- | ------------------ | ---------------- | ------------- |
| Dave  | dave               | google           | com           |
| Steve | steve              | gmail            | com           |
| Rob   | rob                | gmail            | com           |
| Wes   | NaN                | NaN              | NaN           |

---

##### ✅ Now — What is **Vectorized Element Retrieval**?

This line:

```python
matches.str.get(1)
```

means:

> “From every row → retrieve element at position 1”

Since each row contains a collection (like tuple/list/string result),

Pandas **retrieves that element for the whole column at once.**

No loop needed.

Equivalent slow Python logic:

```python
result = []
for row in matches:
    result.append(row[1])
```

Vectorized version:

```python
matches.str.get(1)
```

---

##### ✅ Result

```
Dave     google
Steve    gmail
Rob      gmail
Wes      NaN
dtype: object
```

So:

👉 You **retrieved the domain part of all emails in one vectorized operation.**

---

##### ✅ Why This is Powerful (Senior DS Insight)

Vectorized retrieval gives you:

* 🚀 Much faster execution (C / NumPy level)
* 📉 Less memory overhead
* ✨ Cleaner readable pipelines
* 📊 Natural fit for ML feature engineering

Example real DS usage:

```
Extract email domain → one-hot encode → use as ML feature
```

---

##### ✅ Mental Model (Very Important)

Think:

```
Python loop → element by element

Vectorized pandas → column as one mathematical object
```

So:

> `matches.str.get(1)`
> = “Give me the 2nd captured value from every row simultaneously.”

This is **vectorized element retrieval.**

---


To access elements in the embedded lists, we can pass an index to either of these
functions:

In [142]:
data[:5]

Dave     dave@google.com
Steve    steve@gmail.com
Rob        rob@gmail.com
Wes                  NaN
dtype: object

In [143]:
data.str[:5]

Dave     dave@
Steve    steve
Rob      rob@g
Wes        NaN
dtype: object

See Table 7-5 for more pandas string methods.

In [144]:
columns = ["Method", "Description"]

rows = [
["cat","Concatenate strings element-wise with optional delimiter"],
["contains","Return boolean array if each string contains pattern/regex"],
["count","Count occurrences of pattern"],
["extract","Use regex groups to extract strings; result is a DataFrame with one column per group"],
["endswith","Equivalent to x.endswith(pattern) for each element"],
["startswith","Equivalent to x.startswith(pattern) for each element"],
["findall","Compute list of all occurrences of pattern/regex for each string"],
["get","Index into each element (retrieve i-th element)"],
["isalnum","Equivalent to built-in str.isalnum"],
["isalpha","Equivalent to built-in str.isalpha"],
["isdecimal","Equivalent to built-in str.isdecimal"],
["isdigit","Equivalent to built-in str.isdigit"],
["islower","Equivalent to built-in str.islower"],
["isnumeric","Equivalent to built-in str.isnumeric"],
["isupper","Equivalent to built-in str.isupper"],
["join","Join strings in each element with passed separator"],
["len","Compute length of each string"],
["lower / upper","Convert cases; equivalent to x.lower() or x.upper()"],
["match","Use re.match with regex on each element"],
["pad","Add whitespace to left, right, or both sides"],
["center","Equivalent to pad(side='both')"],
["repeat","Duplicate values (e.g., s.str.repeat(3))"],
["replace","Replace occurrences of pattern/regex"],
["slice","Slice each string"],
["split","Split strings on delimiter or regex"],
["strip","Trim whitespace from both sides"],
["rstrip","Trim whitespace on right side"],
["lstrip","Trim whitespace on left side"]
]

render_book_table(
    "Table 7-5. Partial Listing of Vectorized String Methods",
    columns,
    rows
)

Method,Description
cat,Concatenate strings element-wise with optional delimiter
contains,Return boolean array if each string contains pattern/regex
count,Count occurrences of pattern
extract,Use regex groups to extract strings; result is a DataFrame with one column per group
endswith,Equivalent to x.endswith(pattern) for each element
startswith,Equivalent to x.startswith(pattern) for each element
findall,Compute list of all occurrences of pattern/regex for each string
get,Index into each element (retrieve i-th element)
isalnum,Equivalent to built-in str.isalnum
isalpha,Equivalent to built-in str.isalpha


## 7.4 Conclusion

Effective data preparation can significantly improve productive by enabling you to
spend more time analyzing data and less time getting it ready for analysis. We have
explored a number of tools in this chapter, but the coverage here is by no means comprehensive.
In the next chapter, we will explore pandas’s joining and grouping functionality.